In [ ]:
import os
import s3fs
import pandas as pd

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}
)

fs.ls("acharki-ensae")

In [ ]:
chemin_bpe = "acharki-ensae/Projet-analyse-BPE/BPE25.parquet"

with fs.open(chemin_bpe, "rb") as f:
    bpe = pd.read_parquet(f)

bpe.shape

In [ ]:
bpe.columns

In [ ]:
colonnes = [
    "DOM", "SDOM", "TYPEQU", "SIRET", "NOMRS", "APET",
    "DEPCOM", "DEP", "REG", "LIBCOM",
    "LAMBERT_X", "LAMBERT_Y", "EPSG", "LONGITUDE", "LATITUDE",
    "QUALITE_XY", "QUALITE_GEOLOC",
    "DENS3", "DENS7",
]

bde = bpe[colonnes].copy()

In [ ]:
bpe = bpe.loc[bpe["SDOM"].isin(["B1", "B2"]), :].copy()

In [ ]:
bpe = bpe[bpe["TYPEQU"] != "B103"].copy()

In [ ]:
bpe["TYPEQU"].value_counts().sort_index()

In [ ]:
import pandas as pd
import geopandas as gpd

colonnes = [
    "DOM", "SDOM", "TYPEQU", "SIRET", "NOMRS", "APET",
    "DEPCOM", "DEP", "REG", "LIBCOM",
    "LAMBERT_X", "LAMBERT_Y", "EPSG", "LONGITUDE", "LATITUDE",
    "QUALITE_XY", "QUALITE_GEOLOC",
    "DENS3", "DENS7",
]

bpe_alim = bpe[colonnes].copy()
print("Lignes au départ :", len(bpe_alim))


In [ ]:

for col in ["DOM", "SDOM", "TYPEQU", "SIRET", "DEPCOM", "DEP", "REG", "APET"]:
    bpe_alim[col] = bpe_alim[col].astype("string")


for col in ["LAMBERT_X", "LAMBERT_Y", "LONGITUDE", "LATITUDE"]:
    bpe_alim[col] = pd.to_numeric(bpe_alim[col], errors="coerce")

bpe_alim.dtypes

In [ ]:
bpe_alim.isna().sum()

In [ ]:
sans_coord = bpe_alim["LONGITUDE"].isna() | bpe_alim["LATITUDE"].isna()
bpe_alim.loc[sans_coord, "DEP"].value_counts()

In [ ]:
(bpe_alim["DEP"] == "75").sum()

In [ ]:
# Ajout du code postal à bpe_alim
bpe_alim["CODPOS"] = bpe.loc[bpe_alim.index, "CODPOS"].astype("string")

# Tableau par code postal parisien
paris = bpe_alim[bpe_alim["DEP"] == "75"].copy()
paris["sans_coord"] = paris["LONGITUDE"].isna()

impact_cp = (
    paris.groupby("CODPOS")
    .agg(nb_commerces=("sans_coord", "size"), nb_sans_coord=("sans_coord", "sum"))
    .assign(part_sans_coord=lambda d: (d["nb_sans_coord"] / d["nb_commerces"]).round(3))
    .sort_index()
)

impact_cp

In [ ]:
n_avant = len(bpe_alim)
bpe_alim = bpe_alim.dropna(subset=["LONGITUDE", "LATITUDE"]).copy()

print("Commerces sans coordonnées supprimés :", n_avant - len(bpe_alim))
print("Commerces restants :", len(bpe_alim))

In [ ]:
bpe_alim = bpe_alim[bpe_alim["TYPEQU"] != "B103"].copy()

In [ ]:
# DROM : systèmes de coordonnées différents, on se limite à la métropole
DROM = ["971", "972", "973", "974", "976"]
# Absents de DVF (Alsace-Moselle) : inutiles pour le croisement
ABSENTS_DVF = ["57", "67", "68"]

n_avant = len(bpe_alim)
bpe_alim = bpe_alim[~bpe_alim["DEP"].isin(DROM + ABSENTS_DVF)].copy()
print("Lignes retirées (DROM + Alsace-Moselle) :", n_avant - len(bpe_alim))
print("Lignes restantes :", len(bpe_alim))

In [ ]:
import geopandas as gpd

bpe_geo = gpd.GeoDataFrame(
    bpe_alim,
    geometry=gpd.points_from_xy(bpe_alim["LONGITUDE"], bpe_alim["LATITUDE"]),
    crs="EPSG:4326",
)

bpe_geo = bpe_geo.to_crs("EPSG:2154")

bpe_geo.head()

In [ ]:
print(bpe_geo["DEP"].isin(["971", "972", "973", "974", "976", "57", "67", "68"]).sum())  # doit valoir 0
print(bpe_geo["TYPEQU"].value_counts().sort_index())

In [ ]:
print(bpe_geo["DEP"].isin(["971", "972", "973", "974", "976", "57", "67", "68"]).sum())  # doit valoir 0
print(bpe_geo["TYPEQU"].value_counts().sort_index())

In [ ]:
bpe_geo.plot(markersize=0.1, figsize=(8, 8))

In [ ]:
import os
import s3fs

# Désactive les sommes de contrôle automatiques, mal gérées par le stockage du Datalab
os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]},
    skip_instance_cache=True,
)

In [ ]:
chemin_sortie = "acharki-ensae/Projet-analyse-BPE/processed/bpe_alim_2025.parquet"

with fs.open(chemin_sortie, "wb") as f:
    bpe_geo.to_parquet(f)

fs.ls("acharki-ensae/Projet-analyse-BPE/processed")

In [ ]:
bpe_geo.to_parquet("/home/onyxia/work/bpe_alim_2025.parquet")
